In [6]:

from pathlib import Path
from itertools import combinations
import gzip

data_folder = Path("Data")

# Find all existing compressed participant files
all_files = sorted(data_folder.rglob("subject*.csv.gz"))

# Count observations in each participant file
subject_counts = {}

for file in all_files:
    subject = file.name.replace(".csv.gz", "")

    if subject in subject_counts:
        raise ValueError(f"Duplicate participant: {subject}")

    with gzip.open(file, "rt") as f:
        rows = sum(1 for line in f)

    subject_counts[subject] = rows

assert len(subject_counts) == 9, "Expected 9 participants"

total_rows = sum(subject_counts.values())

# Test every possible combination of 2 participants
results = []

for pair in combinations(sorted(subject_counts), 2):
    reporting_rows = sum(subject_counts[s] for s in pair)
    reporting_pct = reporting_rows / total_rows * 100

    results.append((
        abs(reporting_pct - 20),
        pair,
        reporting_pct
    ))

# Find pair closest to 20%
results.sort()

print("Top 5 possible reporting sets:\n")

for difference, pair, pct in results[:5]:
    print(
        f"{pair}: Reporting {pct:.2f}%, "
        f"Training {100-pct:.2f}%"
    )

best_pair = results[0][1]

print("\nBEST REPORTING PARTICIPANTS:", best_pair)
print(f"Reporting: {results[0][2]:.2f}%")
print(f"Training: {100-results[0][2]:.2f}%")


Top 5 possible reporting sets:

('subject103', 'subject104'): Reporting 20.28%, Training 79.72%
('subject103', 'subject107'): Reporting 19.72%, Training 80.28%
('subject103', 'subject106'): Reporting 21.40%, Training 78.60%
('subject103', 'subject105'): Reporting 21.85%, Training 78.15%
('subject101', 'subject103'): Reporting 21.91%, Training 78.09%

BEST REPORTING PARTICIPANTS: ('subject103', 'subject104')
Reporting: 20.28%
Training: 79.72%


In [7]:
import shutil
import json

new_data_folder = Path("Data_80_20")
new_train = new_data_folder / "training_set"
new_report = new_data_folder / "reporting_set"

# Avoid accidentally overwriting a previous split
if new_data_folder.exists():
    raise FileExistsError(
        "Data_80_20 already exists. Inspect it before continuing."
    )

new_train.mkdir(parents=True)
new_report.mkdir(parents=True)

for file in all_files:
    subject = file.name.replace(".csv.gz", "")

    if subject in best_pair:
        destination = new_report / file.name
    else:
        destination = new_train / file.name

    shutil.copy2(file, destination)

# Record the new split
report_rows = sum(subject_counts[s] for s in best_pair)
train_rows = total_rows - report_rows

split_info = {
    "dataset": "PAMAP2 Protocol, downsampled to 10 Hz",
    "training_participants": [
        s for s in sorted(subject_counts)
        if s not in best_pair
    ],
    "reporting_participants": list(best_pair),
    "training_rows": train_rows,
    "reporting_rows": report_rows,
    "training_percentage": round(
        train_rows / total_rows * 100, 2
    ),
    "reporting_percentage": round(
        report_rows / total_rows * 100, 2
    )
}

with open(new_data_folder / "split_info.json", "w") as f:
    json.dump(split_info, f, indent=4)

print("New split created!")
print(f"Training: {train_rows / total_rows * 100:.2f}%")
print(f"Reporting: {report_rows / total_rows * 100:.2f}%")
print("Saved in:", new_data_folder)


New split created!
Training: 79.72%
Reporting: 20.28%
Saved in: Data_80_20
